# Complete RAG Pipeline: PDF Parsing, Chunking, Embedding, Retrieval & Generation

**Document:** `sample-sdf-document.pdf` (Cytiva Certificate of Quality + storage conditions cover letter, 3 pages)
**Stack:** PyMuPDF (parsing) + LlamaIndex (chunking/indexing/retrieval) + HuggingFace embeddings + Gemini (generation)
**Author:** Bashar

This notebook builds a complete, working RAG pipeline end to end: PDF parsing, chunking, embedding generation, hybrid (vector + keyword) retrieval, and AI response generation with Gemini. It then queries the pipeline with the two required prompts and documents the design choices made at each step.

## Step 0: Install dependencies

In [ ]:
%pip install -q pymupdf llama-index llama-index-llms-google-genai llama-index-embeddings-huggingface llama-index-retrievers-bm25 google-genai nest_asyncio

### A note on Colab's event loop

Colab's notebook kernel already runs its own `asyncio` event loop in the background. LlamaIndex's `.query()` call internally tries to start its own event loop too, which raises `RuntimeError: asyncio.run() cannot be called from a running event loop` if left unpatched. `nest_asyncio.apply()` patches Python's asyncio so a loop can be nested inside another one -- this is a well-known, standard fix for using LlamaIndex (or most async-based libraries) inside Jupyter/Colab, not a workaround specific to this notebook.

In [ ]:
import nest_asyncio
nest_asyncio.apply()

## Step 1: Set up your Gemini API key

In [ ]:
import os
GOOGLE_API_KEY = "YOUR_API_KEY_HERE"  # Replace with your actual API key from Google AI Studio
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY

if GOOGLE_API_KEY == "YOUR_API_KEY_HERE" or not GOOGLE_API_KEY.strip():
    print("WARNING: Please replace YOUR_API_KEY_HERE with your actual API key!")
    print("Get one free at: https://aistudio.google.com/app/apikey")

## Step 2: Upload the PDF

In [ ]:
try:
    from google.colab import files
    print("Please select 'sample-sdf-document.pdf' to upload:")
    uploaded = files.upload()
    pdf_path = list(uploaded.keys())[0]
except ImportError:
    import glob
    candidates = glob.glob("*sdf-document*.pdf") or glob.glob("*.pdf")
    assert candidates, "No PDF found -- please upload sample-sdf-document.pdf"
    pdf_path = candidates[0]

print("Using PDF:", pdf_path)

## Step 3: PDF Parsing

We use PyMuPDF to extract text page by page. Each page becomes one LlamaIndex `Document`, tagged with its page number — this preserves the page boundary, which matters later when citing *where* an answer came from (e.g. "see page 2").

In [ ]:
import fitz  # PyMuPDF
from llama_index.core import Document

def load_pdf_with_pymupdf(path):
    """Parse a PDF into one LlamaIndex Document per page."""
    doc = fitz.open(path)
    documents = []
    for i, page in enumerate(doc):
        text = page.get_text()
        if not text.strip():
            continue
        documents.append(Document(
            text=text,
            metadata={"file_name": os.path.basename(path), "page_number": i + 1, "total_pages": len(doc)}
        ))
    doc.close()
    print(f"Parsed {path}: extracted {len(documents)} pages with content")
    return documents

pdf_docs = load_pdf_with_pymupdf(pdf_path)
for d in pdf_docs:
    print(f"\n--- Page {d.metadata['page_number']} ({len(d.text.split())} words) ---")
    print(d.text[:150].replace(chr(10), " | "), "...")

## Step 4: Chunking strategy

**Choice: page-level chunking**, with LlamaIndex's default `SentenceSplitter` (chunk size 1024 tokens) applied on top as a safety net.

**Why:** this document is only 3 pages, and each page is already a coherent, self-contained unit (page 1 is a storage-conditions cover letter, pages 2–3 are the Certificate of Quality). Splitting mid-page risks cutting a label away from its value (e.g. separating "Storage Temperature" from the "+5°C" that follows it), which would hurt retrieval precision on a document this short. The default splitter only kicks in as a fallback for any page long enough to exceed 1024 tokens — none of these pages are, so in practice each page stays as one chunk. For a much longer document, a smaller chunk size with overlap would make more sense, but for a short compliance letter, page-level chunking keeps each chunk semantically whole.

In [ ]:
from llama_index.core.node_parser import SentenceSplitter

splitter = SentenceSplitter(chunk_size=1024, chunk_overlap=50)
nodes_preview = splitter.get_nodes_from_documents(pdf_docs)
print(f"{len(pdf_docs)} pages in -> {len(nodes_preview)} chunks out (page-level chunking confirmed: 1 page == 1 chunk at this length)")

## Step 5: Embedding model

**Choice: `sentence-transformers/all-MiniLM-L6-v2`** (HuggingFace, run locally).

**Why:** it's small (384 dimensions), fast, and free — it runs entirely on-device with no per-call API cost or external dependency once downloaded, which matters for a document-search tool that might run frequently. For a 3-page compliance letter, a larger/slower embedding model (e.g. `all-mpnet-base-v2`, 768 dimensions) would add latency without a meaningful quality gain at this document size. MiniLM is also one of the most widely used sentence-embedding baselines, so its retrieval behavior is well understood and easy to reason about.

In [ ]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import Settings, VectorStoreIndex
from llama_index.llms.google_genai import GoogleGenAI

embed_model = HuggingFaceEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")
Settings.embed_model = embed_model

llm = GoogleGenAI(model="models/gemini-3.8-flash")  # gemini-2.0-flash was retired by Google; this is its current replacement
Settings.llm = llm

index = VectorStoreIndex.from_documents(pdf_docs)
print(f"Indexed {len(pdf_docs)} pages into the vector store.")

## Step 6: Retrieval method

**Choice: Hybrid retrieval (vector search + BM25 keyword search)**, merged and de-duplicated by score.

**Why:** this is a pharmaceutical compliance document full of exact technical strings that matter character-for-character — "ISO 9001", "EMA/410/01", part numbers like "29-1846-12", temperatures like "+5°C". Vector (semantic) search is good at matching the *meaning* of a natural-language question to a passage, but can under-rank a passage that uses an exact term the embedding model doesn't weight heavily. BM25 keyword search is the opposite: it rewards exact term overlap but has no sense of meaning or paraphrase. Combining both means a question can be answered whether it depends on exact terminology or natural-language phrasing, which matters in a compliance context where both show up.

In [ ]:
from llama_index.retrievers.bm25 import BM25Retriever

def hybrid_retrieve(index, query, top_k=3):
    """Combine vector search and BM25 keyword search, de-duplicated and sorted by score."""
    vector_nodes = index.as_retriever(similarity_top_k=top_k).retrieve(query)

    all_index_nodes = [node for node in index.docstore.docs.values()]
    bm25_retriever = BM25Retriever.from_defaults(nodes=all_index_nodes, similarity_top_k=top_k)
    keyword_nodes = bm25_retriever.retrieve(query)

    combined = {}
    for node in vector_nodes + keyword_nodes:
        if node.node_id not in combined or node.score > combined[node.node_id].score:
            combined[node.node_id] = node

    return sorted(combined.values(), key=lambda n: n.score, reverse=True)[:top_k]

# Quick sanity check of the retriever before wiring it into the full pipeline
test_nodes = hybrid_retrieve(index, "What are the storage conditions for this product?")
for i, n in enumerate(test_nodes):
    print(f"{i+1}. (score={n.score:.4f}, page={n.metadata.get('page_number')}) {n.get_text()[:120]}...")

## Step 7: Response generation — wire retrieval + Gemini into one pipeline

In [ ]:
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.retrievers import BaseRetriever
from llama_index.core.schema import QueryBundle

class HybridRetriever(BaseRetriever):
    """Wraps hybrid_retrieve() so it can plug into LlamaIndex's query engine."""
    def __init__(self, index, top_k=3):
        self.index = index
        self.top_k = top_k
        super().__init__()

    def _retrieve(self, query_bundle: QueryBundle):
        return hybrid_retrieve(self.index, query_bundle.query_str, top_k=self.top_k)

rag_engine = RetrieverQueryEngine.from_args(
    retriever=HybridRetriever(index, top_k=3),
    llm=llm
)

print("RAG pipeline ready: PDF -> page-level chunks -> MiniLM embeddings -> hybrid retrieval -> Gemini generation")

## Step 8: Query the pipeline with the two required prompts

In [ ]:
prompt_1 = "What test methods were used for quality control?"
response_1 = rag_engine.query(prompt_1)
print("PROMPT 1:", prompt_1)
print("\nRESPONSE 1:\n", response_1)

In [ ]:
prompt_2 = "What are the storage conditions specified in the certificate?"
response_2 = rag_engine.query(prompt_2)
print("PROMPT 2:", prompt_2)
print("\nRESPONSE 2:\n", response_2)

## Design choices summary

- **Embedding model:** `sentence-transformers/all-MiniLM-L6-v2` (HuggingFace) — small, fast, free, runs locally, well-understood baseline. A larger model wasn't worth the extra latency for a 3-page document.
- **Chunking strategy:** page-level chunking (1 PDF page = 1 chunk), with LlamaIndex's default `SentenceSplitter` (1024 tokens) as a fallback for any longer page. Keeps each chunk semantically whole on a short, page-structured document.
- **Retrieval method:** hybrid (vector + BM25 keyword), merged and de-duplicated by score. Chosen because this compliance document mixes exact technical terms (ISO 9001, part numbers, temperatures) with natural-language questions — neither pure vector nor pure keyword search alone reliably covers both.